In [8]:
# Ensure that we are using the correct host
import socket
try:
    assert "gpu" in socket.gethostname()
    print(f"Running on {socket.gethostname()}. All is good!")
except:
    raise RuntimeError(f"Be sure to run on GPU! You are currently running on {socket.gethostname()}")

Running on gpu48.storrs.hpc.uconn.edu. All is good!


In [9]:
import os
import pandas as pd
from pathlib import Path
import csv
from tqdm import tqdm

In [10]:
def analyze_file(filepath):
    """
    Analyze a single dataset file for row completeness.
    Skips first 9 columns and last 2 columns when counting completeness.
    Treats "555" and "888" as empty values.
    
    Returns:
        dict with filename, total_rows, filled_rows, incomplete_rows, empty_rows, empty_cell_percentage
    """
    try:
        # Try reading as TSV first (since your example is TSV)
        try:
            df = pd.read_csv(filepath, sep='\t', low_memory=False)
        except:
            # Fall back to regular CSV
            df = pd.read_csv(filepath, low_memory=False)
        
        total_rows = len(df)
        total_columns = len(df.columns)
        
        # Skip first 9 columns and last 2 columns
        # Only analyze the columns in between
        if total_columns > 11:  # Need at least 12 columns to have something to analyze
            df_analyze = df.iloc[:, 9:-2].copy()  # Skip first 9 and last 2
            analyzed_columns = len(df_analyze.columns)
        elif total_columns > 9:  # Has first 9 but not enough for last 2
            df_analyze = df.iloc[:, 9:].copy()  # Just skip first 9
            analyzed_columns = len(df_analyze.columns)
        else:
            # Not enough columns to skip, analyze nothing
            df_analyze = pd.DataFrame()
            analyzed_columns = 0
        
        if analyzed_columns > 0:
            # Replace "555" and "888" with NaN (treating them as empty)
            df_analyze = df_analyze.replace(['555', '888', 555, 888], pd.NA)
            
            # Count filled rows (all non-null in analyzed columns)
            filled_rows = df_analyze.dropna(how='any').shape[0]
            
            # Count completely empty rows (all null in analyzed columns)
            empty_rows = df_analyze.isna().all(axis=1).sum()
            
            # Count incomplete rows (some filled, some empty in analyzed columns)
            incomplete_rows = total_rows - filled_rows - empty_rows
            
            # Calculate percentage of empty cells
            total_cells = df_analyze.size  # total number of cells in analyzed columns
            empty_cells = df_analyze.isna().sum().sum()  # total number of empty cells
            empty_cell_percentage = (empty_cells / total_cells * 100) if total_cells > 0 else 0
        else:
            filled_rows = 0
            empty_rows = 0
            incomplete_rows = 0
            empty_cell_percentage = 0
        
        return {
            'filename': os.path.basename(filepath),
            'filepath': filepath,
            'total_rows': total_rows,
            'total_columns': total_columns,
            'analyzed_columns': analyzed_columns,
            'filled_rows': filled_rows,
            'incomplete_rows': incomplete_rows,
            'empty_rows': empty_rows,
            'empty_cell_percentage': round(empty_cell_percentage, 2),
            'status': 'success'
        }
        
    except Exception as e:
        return {
            'filename': os.path.basename(filepath),
            'filepath': filepath,
            'total_rows': 0,
            'total_columns': 0,
            'analyzed_columns': 0,
            'filled_rows': 0,
            'incomplete_rows': 0,
            'empty_rows': 0,
            'empty_cell_percentage': 0,
            'status': f'error: {str(e)}'
        }

In [ ]:
data_dir = "/home/rif17002/honors_thesis/ABCD_files"
output_path = "/home/rif17002/honors_thesis/dataset_info"
output_file = "ksads_info.csv"

os.makedirs(output_path, exist_ok=True)

# Find all CSV and TSV files
file_patterns = ['*.csv', '*.tsv', '*.txt']
all_files = []

for pattern in file_patterns:
    all_files.extend(Path(data_dir).glob(pattern))

if not all_files:
    print(f"No CSV/TSV files found in {data_dir}")
    exit

print(f"Found {len(all_files)} files to analyze...\n")

# Analyze each file with progress bar
results = []
for filepath in tqdm(all_files, desc="Analyzing files", unit="file"):
    result = analyze_file(str(filepath))
    results.append(result)

Found 374 files to analyze...



Analyzing files:  20%|█████████████████████▋                                                                                         | 73/374 [01:35<04:10,  1.20file/s]

In [ ]:
# Save results to CSV
if results:
    fieldnames = ['filename', 'filepath', 'total_rows', 'total_columns',
                    'analyzed_columns', 'filled_rows', 'incomplete_rows', 
                    'empty_rows', 'empty_cell_percentage', 'status']
    
    with open(output_path + "/" + output_file, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    
    print(f"\n✓ Analysis complete! Results saved to: {output_file}")
    print(f"\nSummary:")
    print(f"  Total files analyzed: {len(results)}")
    print(f"  Successful: {sum(1 for r in results if r['status'] == 'success')}")
    print(f"  Errors: {sum(1 for r in results if r['status'] != 'success')}")
else:
    print("No results to save.")

In [ ]:
# Path to your full analysis results
input_file = "/home/rif17002/honors_thesis/dataset_info/ksads_info.csv"
output_file = "/home/rif17002/honors_thesis/dataset_info/ksads_filtered.csv"

# Define which files you want to keep
# You can specify by exact filename or by pattern
files_to_keep = [
    'pdem02.txt',
    'abcd_cbcls01.txt',
    'abcd_cbcl01.txt',
    'abcd_ksads01.txt',
    'diff_emotion_reg_p01.txt',
    'opp_defiant_disorder_p01.txt',
    'depressive_disorders01.txt',
    'depressive_disorders_p01.txt',
    'disruptive_mood_dysreg01.txt',
    'disruptive_mood_dysreg_p01.txt',
]

# Read the full CSV and filter rows
filtered_results = []

with open(input_file, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Method 1: Check if filename is in your list
        if row['filename'] in files_to_keep:
            filtered_results.append(row)
        
        # Method 2: Use pattern matching (uncomment if you prefer)
        # if should_keep_file(row['filename']):
        #     filtered_results.append(row)

# Write filtered results to new CSV
if filtered_results:
    fieldnames = filtered_results[0].keys()
    
    with open(output_file, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(filtered_results)
    
    print(f"✓ Filtered results saved to: {output_file}")
    print(f"  Files included: {len(filtered_results)}")
    print("\nIncluded files:")
    for result in filtered_results:
        print(f"  - {result['filename']}")
else:
    print("No matching files found!")